In [1]:
%%capture
import os
import pandas as pd
import numpy as np
!pip3 install pyreadstat
!pip3 install fuzzywuzzy
!pip3 install python-Levenshtein
!pip3 install streamlit
#!pip3 install dtale
!pip3 install deepdiff
from statsmodels.formula.api import ols
#from linearmodels.panel import PanelOLS
# Set the default folder (working directory)
default_path = "/Users/app/Dropbox/Distress_2023/Data-Code-Output"
os.chdir(default_path)

Section 1 - to be added: The code from RAW HMDA as downloaded from HMDA to filtered --> single family (standard filter). Twyla's code. 

Section 2 - collapses lending by banks ([removing agency==7] when available, stopped in 2018 in LAR?) at the MSA, after distinguishing between jumbo and non-jumbo loans. 

Section 3 - merge with PANEL HMDA to get the TOP HOLDER ID 

Section 4 - use TOP hlder ID to merge with Banking Data 

Section 5- loan level data including rejections: to carry analysis based on rejection rate. 

END 

In [ ]:
# # Filter conditions
FILTER_CONDITIONS = {
    "derived_dwelling_category": ["Single Family (1-4 Units):Site-Built"],  # Exclude Manufactured
    "loan_purpose": [1, 31],  # Home Purchase, Refinancing
    "lien_status": [1],  # First Lien
    "derived_loan_product_type": [
        "Conventional:First Lien",
        "FHA:First Lien",
        "VA:First Lien",
        "FSA/RHS:First Lien"
    ],
    "occupancy_type": [1],  # Only Principal Residence
    "action_taken": [1]  # Loan originated
}

# Define columns to exclude
EXCLUDED_COLUMNS = [
    "applicant_race_1", "applicant_race_2", "applicant_race_3",
    "applicant_race_4", "applicant_race_5", "co_applicant_race_1",
    "co_applicant_race_2", "co_applicant_race_3", "co_applicant_race_4",
    "co_applicant_race_5"
]

def efficient_filter_and_save_lar_data(source_folder, output_folder, years, output_format="csv"):
    """
    Efficiently filters and processes HMDA LAR data from ZIP archives for multiple years,
    applying filters and saving the filtered results to a specified format.

    Args:
        source_folder (str): Path to the folder containing ZIP files.
        output_folder (str): Path to save the filtered files.
        years (list): List of years to process.
        output_format (str): Output format, either "csv" or "excel".
    """
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    for year in years:
        # Determine file name based on the year and conventions
        if year <= 2017:
            lar_file = f"hmda_{year}_nationwide_first-lien-owner-occupied-1-4-family-records_labels.zip"
        elif year == 2023:
            lar_file = f"2023_public_lar_csv.zip"
        else:
            lar_file = f"{year}_public_lar_one_year_csv.zip" if year > 2018 else f"{year}_public_lar_three_year_csv.zip"

        zip_file = os.path.join(source_folder, lar_file)
        output_file = os.path.join(output_folder, f"{year}_filtered_lar.{output_format}")

        if os.path.exists(zip_file):
            print(f"\nProcessing {zip_file}...")

            try:
                with zipfile.ZipFile(zip_file, 'r') as zf:
                    for file in zf.namelist():
                        if file.endswith(('.csv', '.txt')):
                            print(f"  Reading file: {file}")
                            with zf.open(file) as extracted_file:
                                chunk_size = 100000  # Process data in chunks
                                filtered_chunks = []

                                # Read in chunks
                                for chunk in pd.read_csv(
                                    extracted_file,
                                    delimiter=',',
                                    chunksize=chunk_size,
                                    low_memory=False
                                ):
                                    # Exclude unwanted columns
                                    chunk = chunk[[col for col in chunk.columns if col not in EXCLUDED_COLUMNS]]

                                    # Apply filters to the chunk
                                    for col, values in FILTER_CONDITIONS.items():
                                        if col in chunk.columns:
                                            chunk = chunk[chunk[col].isin(values)]

                                    # Append non-empty chunks
                                    if not chunk.empty:
                                        filtered_chunks.append(chunk)

                                # Concatenate all filtered chunks
                                if filtered_chunks:
                                    filtered_data = pd.concat(filtered_chunks, ignore_index=True)

                                    # Save filtered data in the specified format
                                    if output_format == "csv":
                                        filtered_data.to_csv(output_file, index=False)
                                    elif output_format == "excel":
                                        filtered_data.to_excel(output_file, index=False, engine='openpyxl')
                                    print(f"  Filtered data saved to {output_file}")
                                else:
                                    print(f"  No matching records found in {file}")
                        else:
                            print(f"  Skipped file: {file} (not a CSV or TXT)")
            except Exception as e:
                print(f"Error processing {zip_file}: {e}")
        else:
            print(f"File {zip_file} does not exist. Skipping.")

output_folder_path = '/Users/twylazhang/Desktop/Bank_Research/HMDA/filtered_lar_data'
years_to_process = list(range(2018, 2024))  # Process data from 2018 to 2023
efficient_filter_and_save_lar_data(source_folder_path, output_folder_path, years_to_process, output_format="csv")

In [ ]:
# Section 2 

A/FURTHER FILTERING OF DATA ()

1- Keep ONLY MSAs 
2- KEEP ONLY BANKS 

B/MERGE WITH LOAN LIMITS / CREATE JUMBO LOAN DUMMY

# YEAR 2012
# ensure we have the MSA numbers and agency codes to perform 1 and 2

In [ ]:
import pandas as pd
import numpy as np

# File paths
data_path = "Data/Processed/HMDA/2012_filtered_lar.csv"
jumbo_path = "Data/Raw/HMDA/jumbocutoffs/FY2012.xls"

# Step 1: Sample 0.5% of the data and clean column names
#df_s = pd.read_csv(data_path, skiprows=lambda x: x > 0 and np.random.rand() > 0.005)
df_s = pd.read_csv(data_path)
df_s.columns = [col.strip().replace(' ', '_') for col in df_s.columns]


# Step 2: check data types of columns... 
# Reaons: See how many missing values. It is normal that msamd will have many missing vals. We want to drop those missing. 
# we want the geo codes to be integer. so we are checking their datatype first 
columns_to_check = ['agency_code', 'msamd', 'state_code', 'county_code', 'loan_amount_000s']
df_s2 = df_s[columns_to_check]

# Summarize the selected columns
summary = []
for col in df_s2.columns:
    col_data = df_s2[col]
    col_summary = {
        'Column Name': col,
        'Data Type': col_data.dtype,  # Data type of the column
        'Unique Values': col_data.nunique(),  # Number of unique values
        'Row Count': len(col_data),  # Total number of rows
        '% Empty': (col_data.isnull().sum() / len(col_data)) * 100  # Percentage of missing values
    }
    summary.append(col_summary)
summary_df = pd.DataFrame(summary)
print(summary_df)

# Step 3: Drop rows with missing values in key columns
# notice that we are back to the main data 
df_s = df_s.dropna(subset=columns_to_check)

# Step 4: Convert geographical codes to integers
df_s['state_code'] = df_s['state_code'].astype(int)
df_s['county_code'] = df_s['county_code'].astype(int)
df_s['msamd'] = df_s['msamd'].astype(int)

# Step 5: Drop rows where agency_code is 7
df_s = df_s[~df_s['agency_code'].isin([7])]

# Step 6: Load jumbo loan limit data and clean column names
jumbo = pd.read_excel(jumbo_path)
jumbo.columns = [col.strip().replace(' ', '_') for col in jumbo.columns]
jumbo = jumbo[['FIPS_State_Code', 'FIPS_County_Code', 'One-Unit_Limit']].rename(columns={
    'FIPS_State_Code': 'state_code',
    'FIPS_County_Code': 'county_code',
    'One-Unit_Limit': 'loan_limit'
})

# Step 7: Merge the sampled data with jumbo loan limits
merged2012 = pd.merge(
    df_s,
    jumbo,
    on=['state_code', 'county_code'],
    how='left',
    indicator=True
)

# Step 8: Add jumbo flag and compute jumbo and nj-values 
merged2012['jumbo'] = (merged2012['loan_amount_000s'] * 1000 > merged2012['loan_limit']).astype(int)
merged2012['loan_amount_j']=merged2012['loan_amount_000s']*merged2012['jumbo']
merged2012['loan_amount_nj']=merged2012['loan_amount_000s']*(1-merged2012['jumbo'])

# Step 10: Group by respondent and MSAMD
df2012 = merged2012.groupby(['respondent_id', 'msamd']).agg({
    'loan_amount_j': 'sum',   # Lending volume
    'loan_amount_nj': 'sum',     # Number of loans
    'applicant_income_000s': 'median'  # Median income
}).reset_index()

# Step 11: Rename columns for clarity
df2012 = df2012.rename(columns={
    'applicant_income_000s': 'income_median'
})

# Step 12: Sort by respondent ID
df2012 = df2012.sort_values(by='respondent_id')



In [119]:
output_path = os.path.join("Data", "Processed","HMDA", "collapsed_non7","collapsed_2012.csv")
df2012.to_csv(output_path, index=False)




## REMEMBER TO DOWNLOAD THE HMDA (MAKE IT AVAILABLE OFFLINE) AS YOU RUN THIS

In [ ]:
import pandas as pd
import numpy as np
import os
import gc  # For memory cleanup

# Base paths
data_base_path = "Data/Processed/HMDA"
jumbo_base_path = "Data/Raw/HMDA/jumbocutoffs"
output_base_path = "Data/Processed/HMDA/collapsed_non7"

# Iterate over years
for year in range(2013, 2018):  # Years from 2013 to 2017
    print(f"Processing Year: {year}")
    
    # File paths
    data_path = os.path.join(data_base_path, f"{year}_filtered_lar.csv")
    jumbo_path = os.path.join(jumbo_base_path, f"FY{year}.xlsx")
    output_path = os.path.join(output_base_path, f"collapsed_{year}.csv")

    # Step 1: Load and clean column names
    df_s = pd.read_csv(data_path)
    df_s.columns = [col.strip().replace(' ', '_') for col in df_s.columns]

    # Step 2: Check data types of key columns
    columns_to_check = ['msamd', 'state_code', 'county_code', 'loan_amount_000s']
    df_s2 = df_s[columns_to_check]

    # Step 3: Drop rows with missing values
    df_s = df_s.dropna(subset=columns_to_check)

    # Step 4: Convert geographical codes to integers
    df_s['state_code'] = df_s['state_code'].astype(int)
    df_s['county_code'] = df_s['county_code'].astype(int)
    df_s['msamd'] = df_s['msamd'].astype(int)

    # Step 5: Drop rows where agency_code is 7
    df_s = df_s[~df_s['agency_code'].isin([7])]

    # Step 6: Load jumbo loan limit data
    jumbo = pd.read_excel(jumbo_path)
    jumbo.columns = [col.strip().replace(' ', '_') for col in jumbo.columns]
    jumbo = jumbo[['FIPS_State_Code', 'FIPS_County_Code', 'One-Unit_Limit']].rename(columns={
        'FIPS_State_Code': 'state_code',
        'FIPS_County_Code': 'county_code',
        'One-Unit_Limit': 'loan_limit'
    })

    # Step 7: Merge the data with jumbo loan limits
    merged = pd.merge(
        df_s,
        jumbo,
        on=['state_code', 'county_code'],
        how='left',
        indicator=True
    )

    # Step 8: Add jumbo flag
    merged['jumbo'] = (merged['loan_amount_000s'] * 1000 > merged['loan_limit']).astype(int)
    merged['loan_amount_j']=merged['loan_amount_000s']*merged['jumbo']
    merged['loan_amount_nj']=merged['loan_amount_000s']*(1-merged['jumbo'])
    
    # Step 10: Group by respondent and MSAMD
    df_year = merged.groupby(['respondent_id', 'msamd']).agg({
    'loan_amount_j': 'sum',   # Lending volume
    'loan_amount_nj': 'sum',     # Number of loans
    'applicant_income_000s': 'median'  # Median income
    }).reset_index()

# Step 11: Rename columns for clarity
    df_year = df_year.rename(columns={
    'applicant_income_000s': 'income_median'
    })

# Step 12: Sort by respondent ID
    df_year = df_year.sort_values(by='respondent_id')

    # Step 13: Export the processed data
    df_year.to_csv(output_path, index=False)
    print(f"Exported: {output_path}")

    # Clear memory
    del df_s, df_s2, jumbo, merged, df_year
    gc.collect()
    print(f"Completed Year: {year}\n")


## STARTING in 2018, agency code not reposted. Conforming limit provided. 

# 'loan_amount_000s' becomes 'loan_amount'
# 'msamd' becomes 'derived_msa_md'
# state code now in letters, but county code is 4-5 digits
# comforming --> "NC" is jumbo (double check that the values are large) 

In [ ]:
import pandas as pd
import numpy as np
import os
import gc  # For memory cleanup

# Base paths
data_base_path = "Data/Processed/HMDA"
jumbo_base_path = "Data/Raw/HMDA/jumbocutoffs"
output_base_path = "Data/Processed/HMDA/collapsed_non7"

# Iterate over years
for year in range(2018, 2024):  # Years from 2018 to 2023
    print(f"Processing Year: {year}")

    # File paths
    data_path = os.path.join(data_base_path, f"{year}_filtered_lar.csv")
    jumbo_path = os.path.join(jumbo_base_path, f"FY{year}.xls")
    output_path = os.path.join(output_base_path, f"collapsed_{year}.csv")

    # Step 1: Load and clean column names
    df_s = pd.read_csv(data_path)
    df_s.columns = [col.strip().replace(' ', '_') for col in df_s.columns]

    # Step 2: Create derived columns
    df_s['loan_amount1'] = df_s['loan_amount'] / 1000
    df_s['msamd'] = df_s['derived_msa_md'].astype(int)
    df_s['jumbo'] = (df_s['conforming_loan_limit'] == 'NC')
    df_s['income1'] = df_s['income']

    df_s['loan_amount_j']=df_s['loan_amount1']*df_s['jumbo']
    df_s['loan_amount_nj']=df_s['loan_amount1']*(1-df_s['jumbo'])

    # Step 3: Rename columns for clarity
    df_year = df_s.groupby(['lei', 'msamd']).agg({
    'loan_amount_j': 'sum',   # Lending volume
    'loan_amount_nj': 'sum',     # Number of loans
    'income': 'median'  # Median income
    }).reset_index()

    # Step 5: Sort by respondent LEI
    df_year = df_year.sort_values(by='lei')

    # Step 6: Export the processed data
    df_year.to_csv(output_path, index=False)
    print(f"Exported: {output_path}")

    # Clear memory
    del df_s, df_year
    gc.collect()
    print(f"Completed Year: {year}\n")


In [ ]:
panel_path = "Data/Processed/HMDA/merged_panel_ts_files/hmda_2018_merged.csv"
df_panel=pd.read_csv(panel_path)

print(df_panel.columns)



In [ ]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/merged_panel_ts_files/hmda_2017_merged.csv"
panel_data=pd.read_csv(path)

duplicates = panel_data['respondent id'].duplicated(keep=False)
if duplicates.any():
        print(f"Warning: Duplicate respondent IDs in panel data for {year}")
        print(panel_data[duplicates])

# Now Import the COLLAPSED data year by year, merge it with the panel to obtain the RSSDID 

# there were some name changes in 2017, then major changes in 2018. so we wil proceed in three steps

In [ ]:
import pandas as pd
import os

# Define base paths
collapsed_base_path = "Data/Processed/HMDA/collapsed_non7"
panel_base_path = "Data/Processed/HMDA/merged_panel_ts_files"

# Initialize an empty list to store yearly merged DataFrames
merged_dataframes = []

# Iterate over the years
for year in range(2012, 2017):  # 2012 to 2016
    print(f"Processing Year: {year}")
    
    # Paths for collapsed data and panel data
    collapsed_path = os.path.join(collapsed_base_path, f"collapsed_{year}.csv")
    panel_path = os.path.join(panel_base_path, f"hmda_{year}_merged.csv")
    
    # Import the collapsed data
    coll_data = pd.read_csv(collapsed_path)
    print(f"Columns in collapsed data for {year}: {coll_data.columns}")
    print(f"Data length for {year}: {len(coll_data)}")
    print(f"# of lenders: {coll_data['respondent_id'].nunique()}")
    print(f"# of MSAs: {coll_data['msamd'].nunique()}")
    
    # Drop missing msamd and respondent_id
    coll_data = coll_data.dropna(subset=['msamd', 'respondent_id'])
    print(f"Data length after dropping missing msamd/respondent_id: {len(coll_data)}")
    
    # Add year and period columns
    coll_data['year'] = year
    coll_data['period'] = f"{year}Q2"
    
    # Import the panel data
    panel_data = pd.read_csv(panel_path)
    print(f"Columns in panel data for {year}: {panel_data.columns}")
    
# Check for duplicates based on 'respondent id'
# Check and drop duplicates while keeping only the first occurrence
    initial_length = len(panel_data)
    panel_data = panel_data.drop_duplicates(subset=['respondent id'], keep='first')
    final_length = len(panel_data)
# Calculate percentage of observations dropped
    percent_dropped = ((initial_length - final_length) / initial_length) * 100
# Display results
    print(f"Duplicates removed: {initial_length - final_length} ({percent_dropped:.2f}% of total)")

    # Select and rename columns in panel data
    panel_data = panel_data[['respondent id', 'respondent name (panel)_x', 'respondent rssd id', 'top holder rssd id']].rename(columns={
        'respondent id': 'respondent_id',
        'respondent name (panel)_x': 'respondent_name',
        'respondent rssd id': 'rssd_sub',
        'top holder rssd id': 'rssd_top'
    })
    
    # Merge collapsed data with panel data
    merged_year = pd.merge(coll_data, panel_data, on='respondent_id', how='left', indicator=True)
    summary = merged_year['_merge'].value_counts()
    print(f"Merge summary for {year}:\n{summary}")
    
    # Drop the _merge column
    merged_year = merged_year.drop(columns=['_merge'])
    
    # Append to the list of DataFrames
    merged_dataframes.append(merged_year)
    print(f"Completed processing for {year}\n")

# Concatenate all yearly DataFrames into a single DataFrame
merged_12_16 = pd.concat(merged_dataframes, ignore_index=True)
print(f"Final merged data length (2012-2016): {len(merged_12_16)}")


In [ ]:
path_2017="Data/Processed/HMDA/collapsed_non7/collapsed_2017.csv"
coll_2017=pd.read_csv(path_2017) #coll stands for collapsed 
print(coll_2017.columns)
print(len(coll_2017))
number_lenders=coll_2017['respondent_id'].nunique()
print("# of lenders,",number_lenders)
number_msas=coll_2017['msamd'].nunique()
print("# of msas,",number_msas)

    # Add year and period columns
coll_2017['year']= 2017
coll_2017['period'] = "2017Q2"

panel_path = "Data/Processed/HMDA/merged_panel_ts_files/hmda_2017_merged.csv"
panel_data = pd.read_csv(panel_path)

    # Check for duplicates based on 'respondent id'
# Check and drop duplicates while keeping only the first occurrence
initial_length = len(panel_data)
panel_data = panel_data.drop_duplicates(subset=['respondent id'], keep='first')
final_length = len(panel_data)
# Calculate percentage of observations dropped
percent_dropped = ((initial_length - final_length) / initial_length) * 100
# Display results
print(f"Duplicates removed: {initial_length - final_length} ({percent_dropped:.2f}% of total)")


panel_data = panel_data[['respondent id', 'respondent name (panel)', 'respondent rssd id', 'top holder rssd id']].rename(columns={
        'respondent id': 'respondent_id',
        'respondent name (panel)': 'respondent_name',
        'respondent rssd id': 'rssd_sub',
        'top holder rssd id': 'rssd_top'
    })
    
    # Merge collapsed data with panel data
merged_year = pd.merge(coll_2017, panel_data, on='respondent_id', how='left', indicator=True)
summary = merged_year['_merge'].value_counts()
print(f"Merge summary for {year}:\n{summary}")
    
    # Drop the _merge column
merged_2017 = merged_year.drop(columns=['_merge'])

print(merged_2017.columns)

In [ ]:
import pandas as pd
import os

# Define base paths
collapsed_base_path = "Data/Processed/HMDA/collapsed_non7"
panel_base_path = "Data/Processed/HMDA/merged_panel_ts_files"

# Initialize an empty list to store yearly merged DataFrames
merged_dataframes = []

# Iterate over the years
for year in range(2018, 2024):  # 2018 to 2023
    print(f"Processing Year: {year}")
    
    # Paths for collapsed data and panel data
    collapsed_path = os.path.join(collapsed_base_path, f"collapsed_{year}.csv")
    panel_path = os.path.join(panel_base_path, f"hmda_{year}_merged.csv")
    
    # Import the collapsed data
    coll_data = pd.read_csv(collapsed_path)
    print(f"Columns in collapsed data for {year}: {coll_data.columns}")
    print(f"Data length for {year}: {len(coll_data)}")
    print(f"# of lenders: {coll_data['lei'].nunique()}")
    print(f"# of MSAs: {coll_data['msamd'].nunique()}")
    
    # Add year and period columns
    coll_data['year'] = year
    coll_data['period'] = f"{year}Q2"
    
    # Import the panel data
    panel_data = pd.read_csv(panel_path)
    print(f"Columns in panel data for {year}: {panel_data.columns}")

# Check and drop duplicates while keeping only the first occurrence
    initial_length = len(panel_data)
    panel_data = panel_data.drop_duplicates(subset=['lei'], keep='first')
    final_length = len(panel_data)
# Calculate percentage of observations dropped
    percent_dropped = ((initial_length - final_length) / initial_length) * 100
# Display results
    print(f"Duplicates removed: {initial_length - final_length} ({percent_dropped:.2f}% of total)")

    # Select and rename columns in panel data
    panel_data = panel_data[['lei', 'respondent_name_x', 'respondent_rssd', 'topholder_rssd']].rename(columns={
        'respondent_name_x': 'respondent_name',
        'respondent_rssd': 'rssd_sub',
        'topholder_rssd': 'rssd_top'
    })
    
    # Merge collapsed data with panel data
    merged_year = pd.merge(coll_data, panel_data, on='lei', how='left', indicator=True)
    summary = merged_year['_merge'].value_counts()
    print(f"Merge summary for {year}:\n{summary}")
    
    # Drop the _merge column
    merged_year = merged_year.drop(columns=['_merge'])
    
    # Append the merged DataFrame to the list
    merged_dataframes.append(merged_year)
    print(f"Completed processing for {year}\n")

# Concatenate all yearly DataFrames into a single DataFrame
merged_18_23 = pd.concat(merged_dataframes, ignore_index=True)
print(f"Final merged data length (2018-2023): {len(merged_18_23)}")


In [ ]:
# Concatenate merged_12_16 and merged_2017
merged_12_23 = pd.concat([merged_12_16, merged_2017,merged_18_23], ignore_index=True)

# Check the result
print(f"Final concatenated data length: {len(merged_12_23)}")
print(merged_12_23.head)


final_path="Data/Processed/HMDA/final_processed_HMDA.csv"
merged_12_23.to_csv(final_path, index=False)

## NOW MERGE THE DATA WITH BANK DATA 

In [ ]:
## this is path for all banks
bank_path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/fdic/callfdicholder_2011Q1_2023Q4.csv"
### sample banks path: 
#bank_path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/bloc_y9_call_fdic_tic_cu_glb_reg_1123.csv"
df_bank=pd.read_csv(bank_path)
print(df_bank['period'].dtype)

#df_bank=df_bank[df_bank['dep_f']]

df_bank['idrssd']=df_bank['idrssd'].astype(int).astype(str)
print(df_bank['idrssd'].dtype)
print(df_bank['period'])

df_bank

In [ ]:
df_hmda=merged_12_23

print(df_hmda['rssd_top'].dtype)
print("length with all,",len(df_hmda))
df_hmda=df_hmda[df_hmda['rssd_top']>0].copy()
print("length with only top info,",len(df_hmda))

df_hmda['rssd_top']=df_hmda['rssd_top'].astype(int).astype(str)
df_hmda['idrssd']=df_hmda['rssd_top']

print(df_hmda['period'])
df_hmda

In [ ]:
hmda_bank=pd.merge(df_hmda,df_bank,on=['idrssd','period'],how='left',indicator=True)
summary=hmda_bank['_merge'].value_counts()
print(summary)

hmda_bank=hmda_bank[hmda_bank['_merge']=='both']



In [168]:
path="Data/Processed/Combined/hmda_bank_2012_2023.csv"
hmda_bank.to_csv(path,index=False)

### MERGE LOAN LEVEL DATA WITH JUMBO CUTOFF 

In [ ]:
import pandas as pd
import numpy as np

# File paths
data_path = "Data/Processed/HMDA/2012_filtered_lar.csv"
jumbo_path = "Data/Raw/HMDA/jumbocutoffs/FY2012.xls"

# Step 1: Sample 0.5% of the data and clean column names
#df_s = pd.read_csv(data_path, skiprows=lambda x: x > 0 and np.random.rand() > 0.005)
df_s = pd.read_csv(data_path)
df_s.columns = [col.strip().replace(' ', '_') for col in df_s.columns]


# Step 2: check data types of columns... 
# Reaons: See how many missing values. It is normal that msamd will have many missing vals. We want to drop those missing. 
# we want the geo codes to be integer. so we are checking their datatype first 
columns_to_check = ['agency_code', 'msamd', 'state_code', 'county_code', 'loan_amount_000s']
df_s2 = df_s[columns_to_check]

# Summarize the selected columns
summary = []
for col in df_s2.columns:
    col_data = df_s2[col]
    col_summary = {
        'Column Name': col,
        'Data Type': col_data.dtype,  # Data type of the column
        'Unique Values': col_data.nunique(),  # Number of unique values
        'Row Count': len(col_data),  # Total number of rows
        '% Empty': (col_data.isnull().sum() / len(col_data)) * 100  # Percentage of missing values
    }
    summary.append(col_summary)
summary_df = pd.DataFrame(summary)
print(summary_df)

# Step 3: Drop rows with missing values in key columns
# notice that we are back to the main data 
df_s = df_s.dropna(subset=columns_to_check)

# Step 4: Convert geographical codes to integers
df_s['state_code'] = df_s['state_code'].astype(int)
df_s['county_code'] = df_s['county_code'].astype(int)
df_s['msamd'] = df_s['msamd'].astype(int)

# Step 5: Drop rows where agency_code is 7
df_s = df_s[~df_s['agency_code'].isin([7])]

# Step 6: Load jumbo loan limit data and clean column names
jumbo = pd.read_excel(jumbo_path)
jumbo.columns = [col.strip().replace(' ', '_') for col in jumbo.columns]
jumbo = jumbo[['FIPS_State_Code', 'FIPS_County_Code', 'One-Unit_Limit']].rename(columns={
    'FIPS_State_Code': 'state_code',
    'FIPS_County_Code': 'county_code',
    'One-Unit_Limit': 'loan_limit'
})

# Step 7: Merge the sampled data with jumbo loan limits
merged2012 = pd.merge(
    df_s,
    jumbo,
    on=['state_code', 'county_code'],
    how='left',
    indicator=True
)

# Step 8: Add jumbo flag and compute jumbo and nj-values 
merged2012['jumbo'] = (merged2012['loan_amount_000s'] * 1000 > merged2012['loan_limit']).astype(int)
merged2012['loan_amount_j']=merged2012['loan_amount_000s']*merged2012['jumbo']
merged2012['loan_amount_nj']=merged2012['loan_amount_000s']*(1-merged2012['jumbo'])




## Loan level data including rejections 

In [ ]:
### 2022 I took the static file for consistency with 2023 (also the year in the 1 year panel shows 2023)
## read a sample of the data to get an idea of the format of the entries

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/LARS/2022_public_lar_csv.csv"
df_2022 = pd.read_csv(data_path, skiprows=lambda x: x > 0 and np.random.rand() > 0.0001)

df_2022

In [ ]:
print(df_2022['county_code'].dtype)

In [144]:
columns_series = pd.Series(df_2022.columns)
columns_series.to_csv("/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Temp/2022columns.csv", index=False, header=False)






In [1]:
import os
import pandas as pd

# Define filter conditions and columns to exclude
FILTER_CONDITIONS = {
    "derived_dwelling_category": ["Single Family (1-4 Units):Site-Built"],  # Exclude Manufactured
    "loan_purpose": [1, 31],  # Home Purchase, Refinancing
    "lien_status": [1],  # First Lien
    "derived_loan_product_type": [
        "Conventional:First Lien"
    ],
    "occupancy_type": [1],  # Only Principal Residence
    "action_taken": [1, 3, 7]  # Approved, Denied, or Preapproval Denied
}
# Separate condition for `derived_msa_md`
MSA_FILTER = lambda df: df['derived_msa_md'].notnull() & (df['derived_msa_md'] != 0)

EXCLUDED_COLUMNS = [
    "applicant_race_2", "applicant_race_3",
    "applicant_race_4", "applicant_race_5", "co_applicant_race_1",
    "co_applicant_race_2", "co_applicant_race_3", "co_applicant_race_4",
    "co_applicant_race_5"
]

# Define input and output folders
source_folder = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/LARS"
output_folder = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Define years to process
#years = list(range(2021, 2024))  # Process data from 2021 to 2023
years= [2018]

# Iterate through years
for year in years:
    # Input and output file paths
    lar_file = f"{year}_public_lar_csv.csv"
    input_file = os.path.join(source_folder, lar_file)
    output_file = os.path.join(output_folder, f"{year}_filtered_lar.csv")

    # Check if input file exists
    if os.path.exists(input_file):
        print(f"\nProcessing {input_file}...")
        try:
            # Process the file in chunks to handle large data
            chunk_size = 100000  # Adjust chunk size as needed
            filtered_chunks = []

            # Read file in chunks
            for chunk in pd.read_csv(
                input_file,
                delimiter=',',
                chunksize=chunk_size,
                low_memory=False
            ):
                # Exclude unwanted columns
                chunk = chunk[[col for col in chunk.columns if col not in EXCLUDED_COLUMNS]]

                # Apply filters to the chunk
                for col, values in FILTER_CONDITIONS.items():
                    if col in chunk.columns:
                        chunk = chunk[chunk[col].isin(values)]

                # Apply the `derived_msa_md` filter
                if 'derived_msa_md' in chunk.columns:
                    chunk = chunk[MSA_FILTER(chunk)]

                # Append non-empty chunks
                if not chunk.empty:
                    filtered_chunks.append(chunk)

            # Concatenate all filtered chunks
            if filtered_chunks:
                filtered_data = pd.concat(filtered_chunks, ignore_index=True)

                # Save filtered data to CSV
                filtered_data.to_csv(output_file, index=False)
                print(f"  Filtered data saved to {output_file}")
            else:
                print(f"  No matching records found in {input_file}")

        except Exception as e:
            print(f"Error processing {input_file}: {e}")
    else:
        print(f"File {input_file} does not exist. Skipping.")



Processing /Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/LARS/2018_public_lar_csv.csv...
  Filtered data saved to /Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/2018_filtered_lar.csv


In [2]:
import pandas as pd
import os

### TO CHANGE , rename the loan limit column in xls: "loan_limit"


# Paths for jumbo limits and filtered LAR data
jumbo_limit_path = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/jumbocutoffs"
lar_data_path = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected"
output_path = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit"

# Ensure output folder exists
os.makedirs(output_path, exist_ok=True)

# Process years 2022 and 2023
#years = [2021, 2022, 2023]
years=[2018]
for year in years:
    print(f"Processing year: {year}")
    
    # Load jumbo loan limits
    jumbo_path = os.path.join(jumbo_limit_path, f"FY{year}.xlsx")
    jumbo_limits = pd.read_excel(jumbo_path)
    
    # Ensure county codes are correctly formatted
    jumbo_limits['FIPS_County_Code'] = jumbo_limits['FIPS County Code'].astype(str).str.zfill(3)
    jumbo_limits['county_code'] = jumbo_limits['FIPS State Code'].astype(int).astype(str) + jumbo_limits['FIPS_County_Code']
    jumbo_limits = jumbo_limits[['county_code','loan_limit']]
    #jumbo_limits = jumbo_limits[['county_code', 'One-Unit\nLimit']].rename(columns={'One-Unit\nLimit': 'loan_limit'})
    jumbo_limits = jumbo_limits.set_index('county_code')  # Index for efficient lookups
    
    # Load filtered LAR data
    lar_file = os.path.join(lar_data_path, f"{year}_filtered_lar.csv")
    filtered_chunks = []

    # Process the LAR data in chunks
    for chunk in pd.read_csv(lar_file, chunksize=100000, low_memory=False):
        # Convert county_code in LAR data to integer, then to string
        chunk['county_code'] = chunk['county_code'].fillna(0).astype(int).astype(str)

        # Map loan limits to each row based on county_code
        chunk['loan_limit'] = chunk['county_code'].map(jumbo_limits['loan_limit'])
        
        # Filter rows where loan_amount falls within [0.5 * limit, 2 * limit]
        within_range = (chunk['loan_amount'] >= 0.5 * chunk['loan_limit']) & \
                       (chunk['loan_amount'] <= 2 * chunk['loan_limit'])
        chunk = chunk[within_range]
        
        # Append filtered chunk if not empty
        if not chunk.empty:
            filtered_chunks.append(chunk)

    # Combine and save the filtered data
    if filtered_chunks:
        final_filtered_data = pd.concat(filtered_chunks, ignore_index=True)
        output_file = os.path.join(output_path, f"{year}_filtered_by_jumbo_limit.csv")
        final_filtered_data.to_csv(output_file, index=False)
        print(f"Filtered data for {year} saved to {output_file}")
    else:
        print(f"No matching records found for {year}")


Processing year: 2018
Filtered data for 2018 saved to /Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2018_filtered_by_jumbo_limit.csv


In [5]:
import pandas as pd

# Load the 2021, 2022 and 2023 filtered datasets


path2018 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2018_filtered_by_jumbo_limit.csv"
df_2018 = pd.read_csv(path2018)
df_2018['year'] = 2018  # Add a 'year' column for clarity


path2019 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2019_filtered_by_jumbo_limit.csv"
df_2019 = pd.read_csv(path2019)
df_2019['year'] = 2019  # Add a 'year' column for clarity

path2020 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2020_filtered_by_jumbo_limit.csv"
df_2020 = pd.read_csv(path2020)
df_2020['year'] = 2020  # Add a 'year' column for clarity

path2021 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2021_filtered_by_jumbo_limit.csv"
df_2021 = pd.read_csv(path2021)
df_2021['year'] = 2021  # Add a 'year' column for clarity


path2021 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2021_filtered_by_jumbo_limit.csv"
df_2021 = pd.read_csv(path2021)
df_2021['year'] = 2021  # Add a 'year' column for clarity

path2022 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2022_filtered_by_jumbo_limit.csv"
df_2022 = pd.read_csv(path2022)
df_2022['year'] = 2022  # Add a 'year' column for clarity

path2023 = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2023_filtered_by_jumbo_limit.csv"
df_2023 = pd.read_csv(path2023)
df_2023['year'] = 2023  # Add a 'year' column for clarity

# Concatenate the two datasets
df_combined = pd.concat([df_2018,df_2019,df_2020,df_2021, df_2022, df_2023], ignore_index=True)

# Assign the 'period' column based on the year
df_combined['period'] = df_combined['year'].apply(lambda x: f"{x}Q2")

# Save the combined dataset
output_path = "/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/HMDA/withtherejected/filtered_by_jumbo_limit/2021_2023_combined_filtered_by_jumbo_limit.csv"
df_combined.to_csv(output_path, index=False)

# Print confirmation and summary
print(f"Combined dataset saved to: {output_path}")
print(f"Total rows in combined dataset: {len(df_combined)}")


/var/folders/4h/jrqwm8911h9fjkzfcg0llw800000gq/T/ipykernel_58270/3914975317.py:7: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2018 = pd.read_csv(path2018)
/var/folders/4h/jrqwm8911h9fjkzfcg0llw800000gq/T/ipykernel_58270/3914975317.py:12: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2019 = pd.read_csv(path2019)
/var/folders/4h/jrqwm8911h9fjkzfcg0llw800000gq/T/ipykernel_58270/3914975317.py:16: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2020 = pd.read_csv(path2020)
/var/folders/4h/jrqwm8911h9fjkzfcg0llw800000gq/T/ipykernel_58270/3914975317.py:20: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2021 = pd.read

KeyboardInterrupt: 

In [6]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2018_public_panel_csv.csv"
panel_18=pd.read_csv(path)
panel_18=panel_18[['lei','topholder_rssd']]
panel_18['year']=2018
print(panel_18['topholder_rssd'].dtype)
panel_18 = panel_18[
    (panel_18['topholder_rssd'] != -1) &
    (panel_18['topholder_rssd'] != 0) &
    (panel_18['topholder_rssd'].notnull())
]

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2019_public_panel_csv.csv"
panel_19 = pd.read_csv(path)
panel_19 = panel_19[['lei', 'topholder_rssd']]
panel_19['year'] = 2019
print(panel_19['topholder_rssd'].dtype)
panel_19 = panel_19[
    (panel_19['topholder_rssd'] != -1) &
    (panel_19['topholder_rssd'] != 0) &
    (panel_19['topholder_rssd'].notnull())
]

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2020_public_panel_csv.csv"
panel_20=pd.read_csv(path)
panel_20=panel_20[['lei','topholder_rssd']]
panel_20['year']=2020
print(panel_20['topholder_rssd'].dtype)
panel_20 = panel_20[
    (panel_20['topholder_rssd'] != -1) &
    (panel_20['topholder_rssd'] != 0) &
    (panel_20['topholder_rssd'].notnull())
]

int64
int64
int64


In [7]:
### import the reporter panel 

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2021_public_panel_csv.csv"
panel_21=pd.read_csv(path)
panel_21=panel_21[['lei','topholder_rssd']]
panel_21['year']=2021
print(panel_21['topholder_rssd'].dtype)
panel_21 = panel_21[
    (panel_21['topholder_rssd'] != -1) &
    (panel_21['topholder_rssd'] != 0) &
    (panel_21['topholder_rssd'].notnull())
]
panel_21

int64


,lei,topholder_rssd,year
1,5493003T5D4N1CM46J77,4347208,2021
3,549300R2Q0E1T4DWKH51,5272361,2021
6,549300XARD788LSGZW76,1086168,2021
10,2549002A7VC63ZJQY749,2367921,2021
19,549300JRDBJ5RPA6TY76,1245563,2021
...,...,...,...
4327,549300T1ONVEMLQ4B629,1404687,2021
4328,2549006U8SXATH10WI14,1054000,2021
4329,254900GYV99GSRS94462,1084753,2021
4331,549300CMO3QHQGZ2Z096,1427387,2021


In [8]:
### import the reporter panel 

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2022_public_panel_csv.csv"
panel_22=pd.read_csv(path)
panel_22=panel_22[['lei','topholder_rssd']]
panel_22['year']=2022
print(panel_22['topholder_rssd'].dtype)
panel_22 = panel_22[
    (panel_22['topholder_rssd'] != -1) &
    (panel_22['topholder_rssd'] != 0) &
    (panel_22['topholder_rssd'].notnull())
]
panel_22

int64


,lei,topholder_rssd,year
1,B4TYDEB6GKMZO031MB27,1073757,2022
7,AD6GFRVSDT01YPT1CS68,1069778,2022
8,HUX2X73FUCYHUVH1BK78,1068025,2022
14,JJKC32MCHWDI71265Z06,1074156,2022
15,549300SNAY3J7NZEU618,3292253,2022
...,...,...,...
4457,549300TSIYX9RDYWC806,2575230,2022
4458,549300X08QKYUH256I80,5617645,2022
4459,OX3PU53ZLPQKJ4700D47,1119794,2022
4464,549300QB357BUUUV7A56,5727601,2022


In [9]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Raw/HMDA/reporter_panel/2023_public_panel_csv.csv"
panel_23 = pd.read_csv(path)
panel_23 = panel_23[['lei', 'topholder_rssd']]
panel_23['year'] = 2023
print(panel_23['topholder_rssd'].dtype)
panel_23 = panel_23[
    (panel_23['topholder_rssd'] != -1) &
    (panel_23['topholder_rssd'] != 0) &
    (panel_23['topholder_rssd'].notnull())
]
panel_23

int64


,lei,topholder_rssd,year
4,7H6GLXDRUGQFU57RNE97,1039502,2023
5,DRMSV1Q0EKMEXLAU1P80,1132449,2023
7,03D0JEWFDFUS0SEEKG89,1238565,2023
8,5493001XP7X9J8PCH525,3818381,2023
9,5493005QU87D4OE5GZ21,1100813,2023
...,...,...,...
5108,549300EA0U825E61X330,1134313,2023
5109,549300JL8NM8DX0AMB96,1061156,2023
5110,5493005QK4NV0ZZ5EM64,2313544,2023
5111,549300107HDIK6N8E812,2325912,2023


In [11]:
# Concatenate the three datasets
panel_combined = pd.concat([panel_18,panel_19,panel_20,panel_21, panel_22, panel_23], ignore_index=True)
# merge with the df_combined
hmda_holder_1823=pd.merge(df_combined,panel_combined,on=['year','lei'], how='left',indicator=True)
summary=hmda_holder_1823['_merge'].value_counts()
print(summary)
hmda_holder_1823=hmda_holder_1823[hmda_holder_1823['_merge']=='both']
hmda_holder_1823=hmda_holder_1823.drop(columns={'_merge'})
hmda_holder_1823['topholder_rssd']=hmda_holder_1823['topholder_rssd'].astype(int).astype(str)
hmda_holder_1823['idrssd']=hmda_holder_1823['topholder_rssd']
hmda_holder_1823

_merge
left_only     7645933
both          3515821
right_only          0
Name: count, dtype: int64


,activity_year,lei,derived_msa_md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,loan_limit,year,combined_loan_to_value_ratio,period,topholder_rssd,idrssd
15051,2018,7H6GLXDRUGQFU57RNE97,35380,LA,22071,2.207100e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,133.00,598,1525,76,453100.0,2018,NaN,2018Q2,1039502,1039502
15106,2018,7H6GLXDRUGQFU57RNE97,47664,MI,26125,2.612514e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,98.00,1264,1488,52,453100.0,2018,NaN,2018Q2,1039502,1039502
15113,2018,7H6GLXDRUGQFU57RNE97,24340,MI,26081,2.608100e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,85.00,616,1977,76,453100.0,2018,NaN,2018Q2,1039502,1039502
15114,2018,7H6GLXDRUGQFU57RNE97,47664,MI,26125,2.612519e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,206.00,1594,1631,21,453100.0,2018,NaN,2018Q2,1039502,1039502
15115,2018,7H6GLXDRUGQFU57RNE97,47664,MI,26093,2.609373e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Joint,...,117.00,1199,1373,23,453100.0,2018,NaN,2018Q2,1039502,1039502
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11161749,2023,549300YCCZTMBHGTNQ79,19124,TX,48121,4.812102e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,125.23,896,1254,40,726200.0,2023,80.0,2023Q2,3140288,3140288
11161750,2023,549300YCCZTMBHGTNQ79,12420,TX,48453,4.845300e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,61.51,866,1256,43,726200.0,2023,95.0,2023Q2,3140288,3140288
11161751,2023,549300YCCZTMBHGTNQ79,19124,TX,48397,4.839704e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,199.28,984,1017,15,726200.0,2023,80.0,2023Q2,3140288,3140288
11161752,2023,549300YCCZTMBHGTNQ79,23104,TX,48439,4.843911e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,195.46,1468,1525,31,726200.0,2023,80.0,2023Q2,3140288,3140288


In [12]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/hmda_loan_2018_2023.csv"
hmda_holder_1823.to_csv(path,index=False)

In [13]:
## this is path for all banks
bank_path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/fdic/callfdicholder_2011Q1_2023Q4.csv"
### sample banks path: 
#bank_path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/bloc_y9_call_fdic_tic_cu_glb_reg_1123.csv"
df_bank=pd.read_csv(bank_path)
df_bank=df_bank[(df_bank['period']=='2017Q2')|(df_bank['period']=='2018Q2')|(df_bank['period']=='2019Q2')|(df_bank['period']=='2020Q2')|(df_bank['period']=='2021Q2')|(df_bank['period']=='2022Q2')|(df_bank['period']=='2023Q2')]


# Extract year using slicing
df_bank['year'] = df_bank['period'].str[:4]

# Convert to integer (optional)
df_bank['year'] = df_bank['year'].astype(int)

df_bank['idrssd']=df_bank['idrssd'].astype(int).astype(str)

variables = [
    "cash_c", "securities_c", "tier1_capital_c", "cash_securities_c", "liabilities_c",
    "total_loans_c", "assets_c", "net_income_c", "pll_c", "deposits_below_c",
    "deposits_above_c", "reciprocal_deposits_c", "brokered_deposits_c", "brokered_insured_c",
    "unrealized_htm_c", "total_deposits_c", "recip_nonbro_c", "sweep_nonbro_c",
    "listing_nonbro_c", "gov_deposits_c", "bank_deposits_c", "uninsured_time_c", "dep_f",
    "depins_f", "asset_f", "coredep_f", "rwassets_c", "construction_c", "mortgage_c"
]

# Ensure the DataFrame is sorted by idrssd and time (e.g., year or date)
df_bank = df_bank.sort_values(by=["idrssd", "year"])  # Replace "year" with the appropriate time column name

# Create lagged variables
for var in variables:
    df_bank[f"lag_{var}"] = df_bank.groupby("idrssd")[var].shift(1)


#df_bank=df_bank[df_bank['dep_f']]


print(df_bank['idrssd'].dtype)
print(df_bank['period'])




df_bank

object
41089    2017Q2
41093    2018Q2
41097    2019Q2
41101    2020Q2
41105    2021Q2
          ...  
41047    2019Q2
41051    2020Q2
41055    2021Q2
41059    2022Q2
41063    2023Q2
Name: period, Length: 33778, dtype: object


,idrssd,period,cash_c,securities_c,tier1_capital_c,cash_securities_c,liabilities_c,total_loans_c,assets_c,net_income_c,...,lag_gov_deposits_c,lag_bank_deposits_c,lag_uninsured_time_c,lag_dep_f,lag_depins_f,lag_asset_f,lag_coredep_f,lag_rwassets_c,lag_construction_c,lag_mortgage_c
41089,1000276,2017Q2,14682.0,0.0,45564.0,14682.0,250378.0,215882.0,296257.0,865.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41093,1000276,2018Q2,10770.0,0.0,39688.0,10770.0,259286.0,224948.0,306333.0,1224.0,...,1457.0,329.0,21728.0,228550.0,168667.0,296257.0,206822.0,217046.0,6332.0,129453.0
41097,1000276,2019Q2,10967.0,0.0,42007.0,10967.0,260409.0,231913.0,310643.0,1184.0,...,4152.0,79.0,21526.0,245247.0,178353.0,306333.0,220014.0,224387.0,5097.0,138239.0
41101,1000276,2020Q2,30078.0,0.0,44106.0,30078.0,299464.0,248343.0,352682.0,989.0,...,4410.0,4.0,21581.0,242835.0,177644.0,310643.0,221254.0,231616.0,8503.0,139142.0
41105,1000276,2021Q2,34706.0,0.0,46271.0,34706.0,325534.0,243232.0,380151.0,964.0,...,3841.0,4.0,26395.0,285557.0,209203.0,352682.0,254162.0,0.0,5868.0,146275.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41047,999355,2019Q2,3165.0,0.0,1990.0,3165.0,12743.0,11526.0,14733.0,116.0,...,160.0,100.0,0.0,10570.0,10576.0,13355.0,10570.0,10489.0,0.0,185.0
41051,999355,2020Q2,2846.0,0.0,2058.0,2846.0,12529.0,11686.0,14587.0,133.0,...,149.0,100.0,0.0,10534.0,10546.0,14733.0,10534.0,11277.0,0.0,164.0
41055,999355,2021Q2,6360.0,0.0,2164.0,6360.0,15159.0,10981.0,17323.0,189.0,...,149.0,100.0,0.0,11752.0,11769.0,14587.0,11752.0,11481.0,0.0,130.0
41059,999355,2022Q2,6355.0,0.0,2113.0,6355.0,15494.0,11259.0,17607.0,88.0,...,151.0,100.0,253.0,14840.0,14340.0,17323.0,14587.0,10241.0,0.0,116.0


In [201]:
columns_series = pd.Series(df_bank.columns)
columns_series.to_csv("/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Temp/bankcolumns.csv", index=False, header=False)


In [14]:
#### Merge HMDA banks

hmda_banks_1823=pd.merge(hmda_holder_1823,df_bank,on=['idrssd','period'],how='left',indicator=True)
sum=hmda_banks_1823['_merge'].value_counts()
print(sum)

_merge
both          3489079
left_only       26742
right_only          0
Name: count, dtype: int64


In [15]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/hmda_loan_bank_2018_2023.csv"
hmda_banks_1823.to_csv(path,index=False)

In [17]:
path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/hmda_loan_bank_2018_2023.csv"
df=pd.read_csv(path)

/var/folders/4h/jrqwm8911h9fjkzfcg0llw800000gq/T/ipykernel_58270/3662114681.py:2: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,44,92) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path)


In [19]:
# Step 1: Create the new variable `loan_limit_r`
df['loan_limit_r'] = df['loan_amount'] / df['loan_limit']

# Step 2: Filter rows based on the condition
df = df[(df['loan_limit_r'] <= 1.2) & (df['loan_limit_r'] >= 0.8)]

path="/Users/app/Dropbox/Distress_2023/Data-Code-Output/Data/Processed/Combined/hmda_loan_bank_2018_2023_20pct.csv"
df.to_csv(path,index=False)

In [ ]:
## Ignore below 


In [9]:
path="/Users/app/Dropbox/Dealscan/2021jan_2024sept.csv"
df_2022 = pd.read_csv(path, skiprows=lambda x: x > 0 and np.random.rand() > 0.001)

In [10]:
print(df_2022.columns)

path2="/Users/app/Dropbox/Dealscan/2021jan_2024sept_sample.csv"

df_2022.to_csv(path2,index=False)



Index(['Lender_Parent_Name', 'Lender_Parent_Id', 'Lender_Name', 'Lender_Id',
       'Primary_Role', 'Additional_Roles', 'Lender_Commit', 'Lender_Share',
       'Lender_Operating_Country', 'Lender_Parent_Operating_Country',
       ...
       'Swingline', 'Multi_Currency', 'Bid_Option', 'Bankers_Acceptance',
       'Foreign_Exchange', 'Law_Firm_Name', 'Law_Firm_Lender_Primary',
       'Law_Firm_Lender_Other', 'Law_Firm_Borrower_Primary',
       'Law_Firm_Borrower_Other'],
      dtype='object', length=203)
